# Qwen3-TTS on Google Colab

Run Qwen3-TTS with CUDA GPU acceleration on Google Colab.

**Requirements:** A Colab runtime with GPU (T4 or better).

**Setup:** Upload the `Qwen3-TTS_UserFiles` folder to Google Drive at `My Drive/Qwen3-TTS_UserFiles/`, then run all cells in order.

In [ ]:
# === Settings (edit these before running!) === #@title Settings { display-mode: "form" }
MODEL_SIZE = "1.7B"  # @param ["1.7B", "0.6B"]
PRELOAD_CLONE = True    # @param {type:"boolean"}
PRELOAD_DESIGN = False  # @param {type:"boolean"}
PRELOAD_CUSTOM = False  # @param {type:"boolean"}
DEFAULT_MODE = "design"  # @param ["clone", "design", "custom"]
DEFAULT_VOICE_DESC = "A warm, friendly voice with clear articulation"  # @param {type:"string"}
MAX_NEW_TOKENS = 2048  # @param {type:"integer"}
AUTO_LAUNCH_UI = True  # @param {type:"boolean"}

In [ ]:
# === Cell 1: Setup ===
import os, json

# Mount Google Drive (force_remount picks up newly synced files on re-run)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Link project from Drive to expected location
PROJECT_DIR = '/content/drive/My Drive/Qwen3-TTS_UserFiles'
HOME_DIR = os.path.expanduser('~/Qwen3-TTS_UserFiles')

if not os.path.exists(PROJECT_DIR):
    raise FileNotFoundError(
        f"Project not found at '{PROJECT_DIR}'.\n"
        "Upload the Qwen3-TTS_UserFiles folder to 'My Drive/Qwen3-TTS_UserFiles/' in Google Drive."
    )

if not os.path.exists(HOME_DIR):
    os.symlink(PROJECT_DIR, HOME_DIR)
    print(f'Linked {PROJECT_DIR} -> {HOME_DIR}')

# Create output directory (~/Downloads doesn't exist on Colab)
os.makedirs(os.path.expanduser('~/Downloads'), exist_ok=True)

# Install system dependencies
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null

# Install Python dependencies
!pip install -q torch>=2.0 qwen-tts "transformers==4.57.3" flask librosa soundfile numpy pydub requests gradio accelerate bitsandbytes scipy

# Apply settings from Settings cell
config_path = os.path.expanduser('~/Qwen3-TTS_UserFiles/config.json')
with open(config_path) as f:
    config = json.load(f)

config['advanced']['backend'] = 'torch'
config['advanced']['dtype'] = 'float16'
config['advanced']['model_size'] = MODEL_SIZE
config['models']['clone']['load_at_startup'] = PRELOAD_CLONE
config['models']['design']['load_at_startup'] = PRELOAD_DESIGN
config['models']['custom']['load_at_startup'] = PRELOAD_CUSTOM
config['default_voice_description'] = DEFAULT_VOICE_DESC
config.setdefault('generation', {})['max_new_tokens'] = MAX_NEW_TOKENS

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f'Model size: {MODEL_SIZE}')
print(f'Preload: clone={PRELOAD_CLONE}, design={PRELOAD_DESIGN}, custom={PRELOAD_CUSTOM}')
print(f'Max new tokens: {MAX_NEW_TOKENS}')

# Verify GPU
import torch
print(f'\nCUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print('\nSetup complete!')

In [ ]:
# === Cell 2: Start TTS Server ===
import subprocess, time, requests, sys

sys.path.insert(0, os.path.expanduser('~/Qwen3-TTS_UserFiles'))

# Kill any existing server on port 5123
!kill $(lsof -t -i:5123) 2>/dev/null || true

err_log = os.path.expanduser('~/server_stderr.log')
with open(err_log, 'w') as ef:
    server = subprocess.Popen(
        ['python', os.path.expanduser('~/Qwen3-TTS_UserFiles/voice_server.py')],
        stdout=subprocess.PIPE, stderr=ef
    )

# Wait for server to be ready (up to 90 seconds — first run downloads models)
server_url = 'http://127.0.0.1:5123'
print('Starting server...', end='')
for i in range(90):
    if server.poll() is not None:
        print(f'\nServer exited with code {server.returncode}')
        with open(err_log) as f:
            print(f.read()[-3000:])
        raise RuntimeError('Server failed to start — check errors above')
    try:
        resp = requests.get(f'{server_url}/health', timeout=1)
        if resp.status_code == 200:
            print(f'\nServer ready! (took {i+1}s)')
            health = resp.json()
            print(f'  Backend: {health.get("backend", "N/A")}')
            print(f'  Model size: {health.get("model_size", "N/A")}')
            break
    except requests.ConnectionError:
        pass
    print('.', end='', flush=True)
    time.sleep(1)
else:
    print('\nServer did not respond after 90s. Stderr:')
    with open(err_log) as f:
        print(f.read()[-3000:])

# Show GPU memory after model loading
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    free = total - allocated
    print(f'\nGPU Memory: {allocated:.1f} GB used / {total:.1f} GB total ({free:.1f} GB free)')

In [ ]:
# === Cell 3: Launch Gradio UI ===
# Access the UI via the public URL printed below.
if AUTO_LAUNCH_UI:
    from voice_ui import build_ui
    demo = build_ui()
    demo.launch(
        server_name='0.0.0.0',
        share=True,
        allowed_paths=[os.path.expanduser('~/Downloads'), '/tmp'],
    )
else:
    print('Gradio UI skipped (AUTO_LAUNCH_UI = False in Settings).')
    print('You can still generate audio using the TTSClient in the cells below.')

In [ ]:
# === Cell 4: Quick Generation Example (without UI) ===
from voice_client import TTSClient

client = TTSClient()
output = client.generate(
    'Hello from Google Colab! This is Qwen3 TTS running on a GPU.',
    mode=DEFAULT_MODE,
    description=DEFAULT_VOICE_DESC,
    output='colab_test.wav'
)
print(f'Generated: {output}')

# Play in notebook
from IPython.display import Audio
Audio(output)

# Waveform visualization
try:
    import soundfile as sf
    import matplotlib.pyplot as plt
    import numpy as np
    wav, sr = sf.read(output)
    t = np.arange(len(wav)) / sr
    fig, ax = plt.subplots(figsize=(10, 2))
    ax.plot(t, wav, linewidth=0.3)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude')
    ax.set_title('Generated Waveform')
    plt.tight_layout()
    plt.show()
except ImportError:
    pass

In [ ]:
# === Cell 5: Model Management ===
from voice_client import TTSClient

client = TTSClient()
models = client.get_models()

print('Model Status:')
for name, info in models.get('models', {}).items():
    status = 'LOADED' if info.get('loaded') else 'unloaded'
    mem = f" ({info.get('memory_mb', '?')} MB)" if info.get('loaded') else ''
    print(f'  {name}: {status}{mem}')

print(f'\nBackend: {models.get("backend", "N/A")}')
print(f'Model size: {models.get("model_size", "N/A")}')

# Uncomment to load/unload models on demand:
# client.load_model('design')    # Load the design model
# client.load_model('custom')    # Load the custom model
# client.unload_model('clone')   # Free clone model memory

In [ ]:
# === Cell 6: Create Voice Clone === #@title Voice Cloning { display-mode: "form" }
VOICE_NAME = "my_voice"  # @param {type:"string"}
TRANSCRIPT = ""  # @param {type:"string"}

# Upload audio file
from google.colab import files
print('Upload a .wav or .mp3 audio file (5-30 seconds of clear speech):')
uploaded = files.upload()

if uploaded:
    audio_file = list(uploaded.keys())[0]
    print(f'Uploaded: {audio_file}')

    # Ensure clone model is loaded
    from voice_client import TTSClient
    client = TTSClient()
    try:
        client.load_model('clone')
    except Exception:
        pass

    # Create voice prompt
    sys.path.insert(0, os.path.expanduser('~/Qwen3-TTS_UserFiles'))
    from create_custom_voice import create_voice_prompt

    result = create_voice_prompt(
        audio_path=audio_file,
        voice_name=VOICE_NAME,
        transcript=TRANSCRIPT or None,
    )
    print(f'\nVoice prompt created: {VOICE_NAME}')
    print('Use it with: client.generate("text", mode="clone", prompt=f"{VOICE_NAME}.pt")')
else:
    print('No file uploaded.')

In [ ]:
# === Cell 7: Troubleshooting ===
# Run this cell if you encounter errors to see the server log.
with open(os.path.expanduser('~/server_stderr.log')) as f:
    print(f.read()[-3000:])